# 02. 터널 순응 위험도 표 — 밝기 결과 + 동공 모델 잇기

**하는 일**: 07 노트북(71786 사진 → 바깥 밝기)이 낸 **사진별 밝기**를 받아서, 동공 모델에 넣고, 하나의 **터널 순응 위험도 표**로 묶어요.

```
사진 ──07──▶ 바깥 밝기 ──▶ 전밝기·후밝기 (진행 방향) ──▶ 휘도비 A ──▶ 위험도 상·중·하   (등급은 휘도비로)
                                         │
                                         └─▶ 동공 모델 (02): 정상 상태 크기(W&Y) × 시간곡선(01, PsPM) ──▶ 경계부 끝 동공이 메운 몫 · 남는 몫  (설명)
```

**입력** `sunsight_video/data/밝기추정_사진별.csv` (07이 저장, 메인 스레드와 맞춘 형식)
- image_id, source, cls(입구/출구), captured_at, month, sun_alt, 바깥 밝기_정답
- 모델별: 바깥 밝기_*, 안밝기_*, 전밝기_*, 후밝기_*, 휘도비_*, 위험도_* (모델 = 사진+튜닝 · 물리공식 · 정답)
- 입구: 전 = 밖 바깥 밝기 → 후 = 안(280×조광단계) / 출구: 전 = 안 45 → 후 = 밖 바깥 밝기 · 단위 cd/m²
- 파일이 아직 없으면 **예시 밝기 표**(바깥 밝기 500~8,000)로 돌아가요.

**동공 계산**
1. 정상 상태 크기: Watson & Yellott(2012) 공식, 시야 3,000deg² (SpectroSense 사람 단위 CV로 고름, ML과 비슷 → 공식 사용). 나이 20·40·60·70
2. 경계부 끝(80m ≈ 2.9초) 크기: D(2.9초) = D전 + 열림비율 × (D후 − D전). 열림비율은 01 노트북 공식(입구는 τ2 가정 3개)
3. 동공이 메운 몫 = 동공 면적비, **잔여조도비_가정** = 휘도비 ÷ 동공 면적비 (망막이 감당할 몫의 산술값, 시야 손실·사고확률 아님)

**위험도 등급은 휘도비 A로만** (10/15배, 초안). 동공 칸은 '왜 위험한가'를 설명하는 칸이에요.

In [ ]:
import numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
HERE = Path.cwd()
ROOT = next(p for p in [HERE, *HERE.parents] if (p / 'sunsight_data').exists())
OUT = ROOT / 'sunsight_pupil' / '결과'
BRIGHT = ROOT / 'sunsight_video' / 'data' / '밝기추정_사진별.csv'
par = pd.read_csv(OUT / '동공시간_공식숫자.csv', encoding='utf-8-sig')
P = par[par.대상 == '전체 평균'].set_index('전환')
V_KMH, ZONE_M = 100, 80; T_ZONE = ZONE_M / (V_KMH / 3.6)
AGES = [20, 40, 60, 70]; AREA = 3000; TAU2 = {'빠름': 10, '기본': 30, '느림': 60}
print('밝기 파일:', BRIGHT.exists(), '| 경계부 끝', round(T_ZONE, 2), '초')

In [ ]:
def wy(L, age, area=AREA):
    """Watson & Yellott 2012 동공 지름(mm). L = 휘도 cd/m², 시야 면적 deg², 나이."""
    x = (np.asarray(L, float) * area / 846) ** 0.41
    D = 7.75 - 5.75 * x / (x + 2)
    return D + (age - 28.58) * (0.02132 - 0.009562 * D)
def frac_dark(t, tau2):
    r = P.loc['회색→0']; x = max(t - r.d, 0); A2 = r.b * tau2
    return (r.A1 * (1 - np.exp(-x / r.tau1)) + A2 * (1 - np.exp(-x / tau2))) / (r.A1 + A2)
def frac_bright(t):
    r = P.loc['회색→1']; x = max(t - r.d, 0); return 1 - np.exp(-x / r.tau1)
def ratio_from_L20(L20, cls):            # 07과 같은 식 (입구 지침 조광 4단계, 출구 45)
    step = np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)
    return np.where(np.asarray(cls) == '출구', L20 / 45, L20 / (280 * step))
def grade(R): return np.select([R < 10, R <= 15], ['하', '중'], '상')
print('열림비율 (경계부 끝):', {k: round(float(frac_dark(T_ZONE, v)), 2) for k, v in TAU2.items()}, '| 출구', round(float(frac_bright(T_ZONE)), 2))

## 1. 밝기 읽기 (07 결과 → 긴 형태: 사진 × 모델)

In [ ]:
if BRIGHT.exists():
    b = pd.read_csv(BRIGHT, encoding='utf-8-sig', dtype={'image_id': str})
    if '전밝기_정답' not in b.columns and {'L20_정답', '안밝기_정답'} <= set(b.columns):   # 07은 정답 전·후밝기를 따로 안 줌 → 만들기
        b['전밝기_정답'] = np.where(b.cls == '입구', b.L20_정답, b.안밝기_정답)
        b['후밝기_정답'] = np.where(b.cls == '입구', b.안밝기_정답, b.L20_정답)
    models = [c[len('전밝기_'):] for c in b.columns if c.startswith('전밝기_')]
    keep = [c for c in ['image_id', 'source', 'cls', 'captured_at', 'month', 'sun_alt', 'L20_정답'] if c in b.columns]
    parts = []
    for m in models:
        x = b[keep].copy(); x['밝기모델'] = m
        for k in ['L20', '전밝기', '후밝기', '휘도비', '위험도']:
            x[k] = b.get(f'{k}_{m}')
        parts.append(x)
    B = pd.concat(parts, ignore_index=True); 입력 = '07 사진별 밝기'
else:
    L = np.array([500, 1000, 2000, 3000, 4000, 5000, 6000, 8000], float)
    B = pd.DataFrame([dict(image_id=f'예시_{int(l)}_{c}', cls=c, 밝기모델='예시', L20=l) for l in L for c in ['입구', '출구']])
    in_L = np.where(B.cls == '입구', 280 * np.clip(np.ceil(B.L20 / 4000 * 4) / 4, 0.25, 1), 45)
    B['전밝기'] = np.where(B.cls == '입구', B.L20, 45); B['후밝기'] = np.where(B.cls == '입구', in_L, B.L20)
    B['휘도비'] = ratio_from_L20(B.L20.values, B.cls.values); B['위험도'] = grade(B.휘도비.values); 입력 = '예시 밝기 (07 결과 없음)'
print(입력, '|', len(B), '줄 |', B.밝기모델.value_counts().to_dict())
# 07 등급과 같은 식인지 확인 (휘도비 = 전/후 중 큰 쪽 ÷ 작은 쪽)
chk = np.maximum(B.전밝기, B.후밝기) / np.minimum(B.전밝기, B.후밝기)
print('휘도비 식 일치:', np.allclose(chk, B.휘도비, rtol=5e-3, equal_nan=True))   # 07이 소수 둘째 자리로 반올림해서 여유 0.5%

In [ ]:
# ===== 11/6 주 모델: 10의 '추정 L20 (주 모델)' (직선 회귀) 을 동공 입력 밝기로 =====
G2 = ROOT / 'sunsight_video' / 'data' / '등급2단계_사진별.csv'
def rows_from_L20(df, L20, name):
    x = df[['image_id', 'cls'] + [c for c in ['captured_at', 'month', 'sun_alt'] if c in df.columns]].copy()
    x['밝기모델'] = name; x['L20'] = np.asarray(L20, float)
    step = np.clip(np.ceil(x.L20 / 4000 * 4) / 4, 0.25, 1); inn = np.where(x.cls == '입구', 280 * step, 45)
    x['전밝기'] = np.where(x.cls == '입구', x.L20, 45); x['후밝기'] = np.where(x.cls == '입구', inn, x.L20)
    x['휘도비'] = ratio_from_L20(x.L20.values, x.cls.values); x['위험도'] = grade(x.휘도비.values)
    return x
if G2.exists():
    g10 = pd.read_csv(G2, encoding='utf-8-sig', dtype={'image_id': str})
    if '추정 L20 (주 모델)' in g10.columns:
        B = B[B.밝기모델 != '예시']
        add_parts = [rows_from_L20(g10, g10['추정 L20 (주 모델)'], '주모델(직선회귀)')]
        if '정답' not in set(B.밝기모델) and '정답 L20' in g10.columns:
            add_parts.append(rows_from_L20(g10, g10['정답 L20'], '정답'))
        B = pd.concat([B] + add_parts, ignore_index=True); 입력 += ' + 10 주 모델 추정 밝기'
        print('10 주 모델 추정 밝기 추가 →', B.밝기모델.value_counts().to_dict())
else:
    print('10 결과 없음 → 07 밝기만 사용')

In [ ]:
ONLY_ENTRANCE = True   # 11/5 결정: 암순응(입구)만 봄 → 출구 줄은 결과표·요약에서 뺌
if ONLY_ENTRANCE:
    B = B[B.cls == '입구'].reset_index(drop=True)
print('입구만:', len(B), '줄 |', B.밝기모델.value_counts().to_dict())

## 2. 동공 넣기 (나이 4개 × τ2 가정 3개)

In [ ]:
rows = []
for age in AGES:
    for tname, tau2 in TAU2.items():
        x = B.copy(); x['나이'] = age; x['τ2가정'] = tname
        D_pre, D_post = wy(x.전밝기, age), wy(x.후밝기, age)
        f = np.where(x.cls == '입구', frac_dark(T_ZONE, tau2), frac_bright(T_ZONE))
        D_zone = D_pre + f * (D_post - D_pre)
        x['동공_전_mm'], x['동공_후_mm'], x['동공_경계부끝_mm'] = D_pre, D_post, D_zone
        x['열림비율_경계부끝'] = f
        x['동공면적비_정상'] = np.maximum(D_pre, D_post) ** 2 / np.minimum(D_pre, D_post) ** 2
        x['동공면적비_경계부끝'] = np.maximum(D_pre, D_zone) ** 2 / np.minimum(D_pre, D_zone) ** 2
        x['잔여조도비_가정_정상'] = x.휘도비 / x.동공면적비_정상
        x['잔여조도비_가정_경계부끝'] = x.휘도비 / x.동공면적비_경계부끝
        rows.append(x)
R = pd.concat(rows, ignore_index=True)
R.to_csv(OUT / '터널순응_위험도표.csv', index=False, encoding='utf-8-sig')
print('저장:', OUT / '터널순응_위험도표.csv', len(R), '줄')

## 2-1. 주 모델 등급 붙이기 (10 노트북, 11/6 결정)
주 모델 = **직선 회귀로 바깥 밝기 추정 → 그 추정 밝기를 변수로 더한 2단계 로지스틱**.
- 동공 칸: 10의 `추정 바깥 밝기 (주 모델)`을 밝기 입력으로 씀 (밝기모델 = '주모델(직선회귀)'). 07 밝기·정답 밝기는 비교용으로 같이 계산
- 등급: 10의 `주 모델: 추정 밝기 + 2단계 로지스틱`을 image_id로 붙여 `위험도_주모델`
- 약점 한 줄: 주 모델 등급과 추정밝기 등급이 얼마나 같은지 (입구)

In [ ]:
if G2.exists():
    g = pd.read_csv(G2, encoding='utf-8-sig', dtype={'image_id': str})
    main_col = '주 모델: 추정 밝기 + 2단계 로지스틱' if '주 모델: 추정 밝기 + 2단계 로지스틱' in g.columns else next(c for c in g.columns if c.startswith('2단계 로지스틱'))
    keep = {'image_id': 'image_id', main_col: '위험도_주모델', '정답 등급': '위험도_정답_10', '추정밝기 등급 (참고)': '위험도_추정밝기_10'}
    g = g[[c for c in keep if c in g.columns]].rename(columns=keep)
    R = R.drop(columns=[c for c in keep.values() if c != 'image_id' and c in R.columns]).merge(g, on='image_id', how='left')
    print('10 등급 붙음:', round(R.위험도_주모델.notna().mean(), 3), '| 주 모델 열:', main_col)
    one = R[(R.나이 == 40) & (R.τ2가정 == '기본')].drop_duplicates('image_id')
    ent = one[one.cls == '입구']
    if '위험도_추정밝기_10' in R.columns:
        print(f'약점: 입구 {len(ent)}장에서 주 모델 등급 = 추정밝기 등급 일치율 {(ent.위험도_주모델 == ent.위험도_추정밝기_10).mean():.2f}'
              ' (동공 칸은 추정 밝기로 계산, 등급은 주 모델 → 둘이 다른 사진은 설명이 등급과 어긋날 수 있음)')
    if '위험도_정답_10' in R.columns:
        print(f'입구 주 모델 등급 정답 일치율 {(ent.위험도_주모델 == ent.위험도_정답_10).mean():.2f}')
    R.to_csv(OUT / '터널순응_위험도표.csv', index=False, encoding='utf-8-sig')
else:
    R['위험도_주모델'] = np.nan; print('10의 등급2단계_사진별.csv 없음 → 10을 끝까지 Run All 하면 생김')

## 3. 요약 — 한 장으로 보기
- 위험도 분포: 휘도비 등급 (밝기 모델별)
- 동공이 메운 몫: 정상 상태(다 따라왔을 때) vs 경계부 끝(2.9초) — 경계부 끝엔 동공이 덜 따라와서 **남는 몫(망막 부담)이 더 큼**
- 나이: 나이 들수록 동공이 덜 움직여 남는 몫이 큼

In [ ]:
base = R[R.τ2가정 == '기본']
s1 = base[base.나이 == 40].groupby(['밝기모델', 'cls']).위험도.value_counts().unstack(fill_value=0)
s2 = base.groupby(['cls', '나이']).agg(휘도비=('휘도비', 'median'), 동공면적비_정상=('동공면적비_정상', 'median'),
                                     동공면적비_경계부끝=('동공면적비_경계부끝', 'median'),
                                     잔여_정상=('잔여조도비_가정_정상', 'median'), 잔여_경계부끝=('잔여조도비_가정_경계부끝', 'median'))
s3 = R[(R.나이 == 40) & (R.cls == '입구')].groupby('τ2가정').agg(열림비율=('열림비율_경계부끝', 'first'),
        동공면적비_경계부끝=('동공면적비_경계부끝', 'median'), 잔여_경계부끝=('잔여조도비_가정_경계부끝', 'median'))
print('① 위험도 분포 (40세, 휘도비 등급)'); display(s1)
print('② 동공이 메운 몫 vs 남는 몫 (τ2 기본 30초, 중앙값)'); display(s2.round(2))
print('③ 입구 τ2 가정 민감도 (40세)'); display(s3.round(2))
s2.round(3).to_csv(OUT / '터널순응_요약_나이별.csv', encoding='utf-8-sig')

## 3-1. 밝기 오차가 동공 결과로 얼마나 넘어가나
같은 사진에서 **정답 밝기로 계산한 동공 칸** vs **모델 밝기로 계산한 동공 칸**을 비교해요 (40세, τ2 기본). 차이가 작으면 밝기 모델을 그대로 이어 써도 된다는 근거가 돼요.

In [ ]:
from scipy.stats import spearmanr
x = R[(R.나이 == 40) & (R.τ2가정 == '기본')]
if '정답' in set(x.밝기모델):
    t = x[x.밝기모델 == '정답'].set_index('image_id')
    out = []
    for m in sorted(set(x.밝기모델) - {'정답'}):
        p = x[x.밝기모델 == m].set_index('image_id').reindex(t.index)
        for col in ['동공_전_mm', '동공_경계부끝_mm', '동공면적비_경계부끝', '잔여조도비_가정_경계부끝']:
            out.append(dict(밝기모델=m, 칸=col, 평균오차=(p[col] - t[col]).abs().mean(), 정답중앙값=t[col].median(),
                            순위상관=spearmanr(p[col], t[col]).correlation))
        out.append(dict(밝기모델=m, 칸='위험도 등급 일치율', 평균오차=np.nan, 정답중앙값=np.nan, 순위상관=(p.위험도 == t.위험도).mean()))
    prop = pd.DataFrame(out); prop.to_csv(OUT / '터널순응_밝기오차전파.csv', index=False, encoding='utf-8-sig'); display(prop.round(3))
else:
    print('정답 밝기가 없어서 건너뜀')

## 4. 읽는 법 · 한계
- **위험도 = 휘도비 A 등급** (문헌 10/15배 초안). 동공 칸은 등급을 바꾸지 않고 '왜 위험한가'를 설명: 동공이 메우는 몫은 1.x배뿐, 나머지 수 배는 망막이 감당.
- **잔여조도비_가정**은 휘도비 ÷ 동공 면적비의 산술값이에요. 망막 순응량·시야 손실·사고 확률이 아니에요.
- 동공 크기 공식은 균일 시야 가정(E = πL), 시간곡선은 22명 여성 36~49세 실험실 화면, 5초 이후는 τ2 가정 → 숫자는 **범위로** 말하기.
- 밝기 모델이 바뀌면(07 재실행) 이 노트북만 다시 Run All 하면 돼요.